In [ ]:
import os
import glob
import h5py
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
from IPython.display import display, HTML

from vip_slap2_analysis.voltage.extraction import load_voltage_roi_transform_h5
from vip_slap2_analysis.plotting.plot_session_heatmap import _robust_row_zscore
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.summary import VoltageSummary
from vip_slap2_analysis.voltage.postprocess import concat_rois_across_trials
from vip_slap2_analysis.utils.utils import save_figure

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")

In [ ]:
%matplotlib notebook

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
today_str = datetime.today().strftime('%Y-%m-%d')

BASE_PATH = Path(r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics')
SAVE_PATH = Path(r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures')

TARGET_MICE = [
    852835,
]

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

# Voltage extraction writes files like:
#   voltage_session_traces_dff_robust_f0_trial.h5
# This variant controls the filename suffix. The plotted dataset is SIGNAL below.
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL = "dff"      # one of: "raw_f", "f0", "dff"

# Optional direct override. Leave as None to resolve from asset.derived_dir / "voltage".
SESSION_TRACE_H5 = None

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=TARGET_MICE,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions.")
display(process_df)

In [ ]:
asset = assets[-1]
print(asset.session_id)
print("session_dir:", asset.session_dir)
print("derived_dir:", asset.derived_dir)
print("voltage assets:")
for key, path in asset.get_modality_assets("voltage").items():
    print(f"  {key}: {path}")

In [ ]:
vs = VoltageSummary(asset.modality_assets['voltage']['summary_mat'])

## Plot raw F for several traces

In [ ]:
trial = 10
dmd = 1
traces = vs.get_roi_traces(dmd=dmd,trial=trial).T

In [ ]:
from vip_slap2_analysis.voltage.postprocess import concat_rois_across_trials

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

im_rate = 10_800
t = np.linspace((len(traces[0])/im_rate)*trial,(len(traces[0])/im_rate)*(trial+1),len(traces[0]))

for i,d in enumerate(traces[:5]):
    ax.plot(t,d)
    
ax.set_xlabel('Session time (s)')
ax.set_ylabel('Fluorescence')

ax.set_title('Raw ASAP7y dendritic voltage Fluorescence')

fig.tight_layout()
filen = 'rw_F_example'
save_figure(fig,os.path.join(SAVE_PATH,filen),formats = ['.pdf','.png'],dpi=300)

In [ ]:
session_traces = concat_rois_across_trials(vs,dmd=dmd)

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

im_rate = 10_800
t = np.linspace(0,len(session_traces[0][0])/im_rate,len(session_traces[0][0]))
t_window = [0,1800]

for i,d in enumerate(session_traces[0][:5]):
    ax.plot(t[(t>t_window[0])&(t<t_window[1])][::100],d[(t>t_window[0])&(t<t_window[1])][::100],zorder=-i)
    
ax.set_xlabel('Session time (s)')
ax.set_ylabel('Fluorescence')

# ax.set_title('Raw ASAP7y dendritic voltage Fluorescence')

fig.tight_layout()
filen = 'raw_F_example_session'
save_figure(fig,os.path.join(SAVE_PATH,filen),formats = ['.pdf','.png'],dpi=300)

### Plot comparison between F, F0, and dFF for one ROI

In [ ]:
dmd = 1
roi_index = 0
trace_signal = "dff_robust_f0"

session_trace_h5 = asset.derived_dir / 'voltage' / 'voltage_session_traces_dff_robust_f0_trial.h5'
print("Inspecting:", session_trace_h5)

if not session_trace_h5.exists():
    print("Session trace H5 not found. Run voltage extraction first, or use the next cell to compute one ROI on the fly.")
else:
    roi = load_voltage_roi_transform_h5(session_trace_h5, dmd=dmd, roi_index=roi_index)
    t = roi["timebase_sec"]
    raw_f = roi["raw_f"]
    f0 = roi["f0"]
    dff = roi["dff"]
    window = (0,1800)

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    ax.plot(t[t<=window[1]], raw_f[t<=window[1]], lw=0.4, label="raw F")
    ax.plot(t[t<=window[1]], f0[t<=window[1]], lw=2.0, label="F0")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Fluorescence")
    ax.set_title("Example session - Raw F and estimated F$_{0}$")
    ax.legend(frameon=False,fontsize=12)
    fig.tight_layout()
    filen = 'rawF_F0_example'
    save_figure(fig,os.path.join(SAVE_PATH,filen),formats = ['.pdf','.png'],dpi=300)
    
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    ax.plot(t[t<=window[1]], dff[t<=window[1]], lw=0.4)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("\u0394F/F$_{0}$ = (F$_{0}$ - F) / F$_{0}$")
    ax.set_title('\u0394F/F$_{0}$')
    fig.tight_layout()
    filen = 'dFF_example'
    save_figure(fig,os.path.join(SAVE_PATH,filen),formats = ['.pdf','.png'],dpi=300)

### Plot whole-session raw F for all ROIs and DMDs

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from vip_slap2_analysis.voltage.postprocess import concat_rois_across_trials


def _minmax_downsample_2d(X, sample_rate_hz=10_800, max_points=40_000):
    """
    X is time x ROI.
    Preserves the min/max envelope within temporal bins, much better than simple
    striding for voltage traces with brief excursions.
    """
    X = np.asarray(X)
    n_samples, n_rois = X.shape

    if n_samples <= max_points:
        return np.arange(n_samples, dtype=np.float64) / sample_rate_hz, X

    n_bins = max(1, max_points // 2)
    bin_size = max(1, n_samples // n_bins)
    n_use = (n_samples // bin_size) * bin_size

    Xb = X[:n_use].reshape(n_use // bin_size, bin_size, n_rois)

    ymin = np.nanmin(Xb, axis=1)
    ymax = np.nanmax(Xb, axis=1)

    xb = (
        np.arange(ymin.shape[0], dtype=np.float64) * bin_size
        + 0.5 * bin_size
    ) / sample_rate_hz

    x_plot = np.repeat(xb, 2)

    X_plot = np.empty((2 * ymin.shape[0], n_rois), dtype=np.float32)
    X_plot[0::2] = ymin
    X_plot[1::2] = ymax

    return x_plot, X_plot


def _robust_spacing(X_plot, overlap=0.90, min_spacing=1.0):
    """
    Data-driven vertical spacing. Smaller overlap gives more overlap;
    larger overlap gives more separation.
    """
    lo = np.nanpercentile(X_plot, 2, axis=0)
    hi = np.nanpercentile(X_plot, 98, axis=0)
    robust_ranges = hi - lo

    spacing = np.nanpercentile(robust_ranges, 90) * overlap

    if not np.isfinite(spacing) or spacing <= 0:
        spacing = min_spacing

    return max(float(spacing), float(min_spacing))


def plot_concatenated_voltage_rois(
    vs,
    *,
    dmds=(1, 2),
    sample_rate_hz=10_800,
    max_points=40_000,
    time_unit="min",        # "s" or "min"
    spacing=None,           # None = estimate separately per DMD
    overlap=0.90,
    center="median",        # "median", "mean", "first", or None
    drop_discarded=True,
    trace_mode="trial",
    dtype=np.float32,
    roi_height=0.22,        # inches per ROI
    min_axis_height=1.4,
    width=8.0,
    lw=0.45,
    rasterized=True,
):
    loaded = {}
    n_by_dmd = []

    for dmd in dmds:
        X, trial_slices = concat_rois_across_trials(
            vs,
            dmd=dmd,
            drop_discarded=drop_discarded,
            dtype=dtype,
            return_array=True,
            trace_mode=trace_mode,
        )

        # X is time x ROI.
        loaded[int(dmd)] = {
            "X": X,
            "trial_slices": trial_slices,
        }
        n_by_dmd.append(int(X.shape[1]))

    height_ratios = [max(1, n) for n in n_by_dmd]
    fig_height = sum(max(min_axis_height, roi_height * n) for n in n_by_dmd) + 0.8

    fig, axes = plt.subplots(
        len(dmds),
        1,
        figsize=(width, fig_height),
        sharex=True,
        gridspec_kw={"height_ratios": height_ratios},
        constrained_layout=True,
    )


    axes = np.atleast_1d(axes)
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["k"])

    for ax, dmd, n_rois in zip(axes, dmds, n_by_dmd):
        X = loaded[int(dmd)]["X"]

        x_sec, Xp = _minmax_downsample_2d(
            X,
            sample_rate_hz=sample_rate_hz,
            max_points=max_points,
        )

        if time_unit.lower().startswith("min"):
            x = x_sec / 60
            xlabel = "Time (min)"
        else:
            x = x_sec
            xlabel = "Time (s)"

        if center is None:
            centers = np.zeros(n_rois, dtype=float)
        elif center == "median":
            centers = np.nanmedian(Xp, axis=0)
        elif center == "mean":
            centers = np.nanmean(Xp, axis=0)
        elif center == "first":
            centers = np.nanmean(Xp[:min(1000, Xp.shape[0])], axis=0)
        else:
            raise ValueError("center must be 'median', 'mean', 'first', or None")

        X_centered = Xp - centers

        this_spacing = spacing
        if this_spacing is None:
            this_spacing = _robust_spacing(X_centered, overlap=overlap)

        offsets = np.arange(n_rois, dtype=float) * float(this_spacing)

        segments = []
        colors = []

        for r in range(n_rois):
            y = X_centered[:, r] + offsets[r]
            segments.append(np.column_stack((x, y)))
            colors.append(color_cycle[r % len(color_cycle)])

        lc = LineCollection(
            segments,
            colors=colors,
            linewidths=lw,
            rasterized=rasterized,
        )

        ax.add_collection(lc)
        ax.autoscale_view()
        ax.margins(x=0, y=0.03)

        ax.set_yticks(offsets)
        ax.set_yticklabels(np.arange(n_rois))
        ax.set_ylabel(f"DMD{dmd}\nROI")

        ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
        ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
        ax.spines[["top", "right"]].set_visible(False)

    axes[-1].set_xlabel(xlabel)

    return fig, axes, loaded


fig, ax, loaded = plot_concatenated_voltage_rois(
    vs,
    dmds=(1, 2),
    sample_rate_hz=10_800,
    max_points=40_000,
    time_unit="sec",
    overlap=1,
    roi_height=0.15,
    width=5,
)

filen = "raw_example_fast_even_spacing"
save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf", ".png"],
    dpi=300,
)

### Plot traces from several candidate ROIs at multiple timescales

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


def _as_roi_labels(roi_ids, roi_indices):
    labels = []
    for r in roi_indices:
        if roi_ids is None:
            labels.append(str(r))
        else:
            value = roi_ids[r]
            if isinstance(value, bytes):
                value = value.decode()
            labels.append(str(value))
    return labels


def _read_dff_window_h5(
    h5_path,
    *,
    dmd,
    roi_indices,
    window_sec,
    signal="dff",
):
    """Read one H5 time window for selected ROIs; returns time x ROI."""
    dmd_key = f"DMD{int(dmd)}"
    roi_indices = np.asarray(roi_indices, dtype=int)

    with h5py.File(h5_path, "r") as h5:
        g = h5[dmd_key]
        t_all = np.asarray(g["timebase_sec"][:], dtype=float)
        y_ds = g[signal]

        i0 = max(0, int(np.searchsorted(t_all, window_sec[0], side="left")))
        i1 = min(len(t_all), int(np.searchsorted(t_all, window_sec[1], side="right")))

        if i1 <= i0:
            raise ValueError(
                f"{window_sec} does not overlap {dmd_key} timebase "
                f"[{t_all[0]:.3f}, {t_all[-1]:.3f}] s"
            )

        t = t_all[i0:i1]

        if y_ds.shape[-1] == len(t_all):       # ROI x time
            X = y_ds[roi_indices, i0:i1].T
        elif y_ds.shape[0] == len(t_all):      # time x ROI
            X = y_ds[i0:i1, roi_indices]
        else:
            raise ValueError(
                f"Cannot infer orientation for {dmd_key}/{signal}: "
                f"shape={y_ds.shape}, n_time={len(t_all)}"
            )

        roi_ids = g["roi_ids"][:] if "roi_ids" in g else None
        labels = _as_roi_labels(roi_ids, roi_indices)

    return t, np.asarray(X, dtype=np.float32), labels


def _minmax_downsample(t, X, max_points=60_000):
    """Min/max envelope downsampling for time x ROI data."""
    t = np.asarray(t)
    X = np.asarray(X)
    if X.ndim == 1:
        X = X[:, None]

    n_samples, n_rois = X.shape
    if max_points is None or n_samples <= max_points:
        return t, X

    n_bins = max(1, max_points // 2)
    bin_size = max(1, n_samples // n_bins)
    n_use = (n_samples // bin_size) * bin_size

    Xb = X[:n_use].reshape(n_use // bin_size, bin_size, n_rois)
    tb = t[:n_use].reshape(n_use // bin_size, bin_size)

    ymin = np.nanmin(Xb, axis=1)
    ymax = np.nanmax(Xb, axis=1)
    tmid = np.nanmean(tb, axis=1)

    X_plot = np.empty((2 * len(tmid), n_rois), dtype=np.float32)
    X_plot[0::2] = ymin
    X_plot[1::2] = ymax

    return np.repeat(tmid, 2), X_plot


def _robust_spacing(X, multiplier=1.15, min_spacing=0.02):
    ranges = (
        np.nanpercentile(X, 98, axis=0)
        - np.nanpercentile(X, 2, axis=0)
    )
    spacing = np.nanpercentile(ranges, 90) * multiplier

    if not np.isfinite(spacing) or spacing <= 0:
        spacing = min_spacing

    return max(float(spacing), float(min_spacing))


def plot_selected_dff_rois(
    h5_path,
    *,
    dmd_rois,
    window_sec,
    signal="dff",
    event_spans=None,
    image_legend=None,
    max_points=60_000,
    time_unit="s",
    center="median",
    spacing=None,
    spacing_multiplier=1.15,
    dmd_gap=0.8,
    event_alpha=0.28,
    trace_colors=None,
    lw=0.55,
    width=8,
    roi_height=0.32,
    min_height=2.2,
    title=None,
    show_scale_bar=True,
    save_path=None,
    save_name=None,
):
    """
    Plot arbitrary selected ROIs from one or both DMDs.

    dmd_rois
        {1: [roi0, roi1, ...], 2: [roi0, roi1, ...]}

    window_sec
        Window in the H5 trace timebase. The displayed x-axis begins at zero.

    event_spans
        DataFrame returned by load_image_presentation_spans. It must contain
        trace_start_sec, trace_end_sec, and color.
    """
    window_sec = tuple(map(float, window_sec))
    dmd_rois = {
        int(dmd): np.asarray(rois, dtype=int)
        for dmd, rois in dmd_rois.items()
        if rois is not None and len(rois) > 0
    }
    if not dmd_rois:
        raise ValueError("Select at least one ROI")

    traces = []
    for dmd, roi_indices in dmd_rois.items():
        t, X, labels = _read_dff_window_h5(
            h5_path,
            dmd=dmd,
            roi_indices=roi_indices,
            window_sec=window_sec,
            signal=signal,
        )
        for j, label in enumerate(labels):
            traces.append(
                {
                    "dmd": dmd,
                    "roi_index": int(roi_indices[j]),
                    "roi_label": label,
                    "t": t,
                    "y": X[:, j],
                }
            )

    n_groups = len(dmd_rois)
    fig_height = max(
        min_height,
        roi_height * (len(traces) + dmd_gap * max(0, n_groups - 1)) + 1.2,
    )
    fig, ax = plt.subplots(figsize=(width, fig_height))

    # Image presentations/omissions, behind the traces.
    if event_spans is not None:
        for event in event_spans.itertuples():
            start = float(event.trace_start_sec)
            end = float(event.trace_end_sec)

            if end < window_sec[0] or start > window_sec[1]:
                continue

            start = max(start, window_sec[0])
            end = min(end, window_sec[1])

            if time_unit.lower().startswith("min"):
                xs = (start - window_sec[0]) / 60
                xe = (end - window_sec[0]) / 60
            else:
                xs = start - window_sec[0]
                xe = end - window_sec[0]

            ax.axvspan(
                xs,
                xe,
                color=event.color,
                alpha=event_alpha,
                lw=0,
                zorder=0,
            )

    # Downsample and center.
    plotted = []
    for trace in traces:
        tp, yp = _minmax_downsample(
            trace["t"],
            trace["y"][:, None],
            max_points=max_points,
        )
        y = yp[:, 0]

        if center == "median":
            y -= np.nanmedian(y)
        elif center == "mean":
            y -= np.nanmean(y)
        elif center == "first":
            y -= np.nanmean(y[:min(1000, len(y))])
        elif center is not None:
            raise ValueError("center must be median, mean, first, or None")

        plotted.append((tp, y))

    if spacing is None:
        spacing = _robust_spacing(
            np.column_stack([y for _, y in plotted]),
            multiplier=spacing_multiplier,
        )

    if trace_colors is None:
        trace_colors = {1: "#e99c81", 2: "#7bbcd5"}

    y_positions = []
    y_labels = []
    dmd_y = {}
    offset_units = 0.0
    previous_dmd = None

    for trace, (tp, y) in zip(traces, plotted):
        if previous_dmd is not None and trace["dmd"] != previous_dmd:
            offset_units += dmd_gap
        previous_dmd = trace["dmd"]

        offset = offset_units * spacing
        x = tp - window_sec[0]
        if time_unit.lower().startswith("min"):
            x = x / 60

        ax.plot(
            x,
            y - offset,
            color=trace_colors.get(trace["dmd"], "k"),
            lw=lw,
            zorder=2,
            solid_joinstyle="round",
            solid_capstyle="round",
        )

        y_positions.append(-offset)
        y_labels.append(str(trace["roi_label"]))
        dmd_y.setdefault(trace["dmd"], []).append(-offset)
        offset_units += 1.0

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)
    ax.set_ylabel("ROI")
    ax.set_xlabel(
        "Time from window start (min)"
        if time_unit.lower().startswith("min")
        else "Time from window start (s)"
    )

    if title is not None:
        ax.set_title(title)

    # DMD labels sit outside the ROI labels.
    dmd_transform = ax.get_yaxis_transform()
    for dmd, positions in dmd_y.items():
        ax.text(
            -0.075,
            np.mean(positions),
            f"DMD{dmd}",
            transform=dmd_transform,
            rotation=90,
            va="center",
            ha="center",
            color=trace_colors.get(dmd, "k"),
            fontsize=10,
            fontweight="bold",
            clip_on=False,
        )

    if image_legend:
        handles = [
            Patch(
                facecolor=color,
                edgecolor="none",
                alpha=event_alpha,
                label=label,
            )
            for label, color in image_legend.items()
        ]
        ax.legend(
            handles=handles,
            title="Image identity",
            frameon=False,
            ncol=min(4, len(handles)),
            fontsize=8,
            title_fontsize=9,
            loc="lower left",
            bbox_to_anchor=(0, 1.01),
            borderaxespad=0,
        )

    ax.margins(x=0, y=0.05)
    ax.tick_params(axis="x", top=False, labelsize=12)
    ax.tick_params(axis="y", right=False, labelsize=10)
    ax.spines[["top", "right"]].set_visible(False)

    if show_scale_bar:
        scale = spacing / 2
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        xb = x1 - 0.025 * (x1 - x0)
        yb = y0 + 0.06 * (y1 - y0)
        ax.plot([xb, xb], [yb, yb + scale], color="k", lw=1.5, clip_on=False)
        ax.text(
            xb - 0.01 * (x1 - x0),
            yb + scale / 2,
            f"{scale:.3g} dFF",
            va="center",
            ha="right",
            fontsize=8,
        )

    fig.tight_layout()

    if save_path is not None and save_name is not None:
        save_figure(
            fig,
            os.path.join(save_path, save_name),
            formats=[".pdf", ".png"],
            dpi=300,
        )

    return fig, ax, traces

In [ ]:
n_rois = vs.n_rois
n_rois

In [ ]:
RAND_ROI = False

session_trace_h5 = (
    asset.derived_dir
    / "voltage"
    / "voltage_session_traces_dff_robust_f0_trial.h5"
)

roi_idx = [[0,1,2,3],
           [0,1,2]] #specific ROI ids by DMD
num_rois = 4

if RAND_ROI:
    dmd_rois = {
        1: np.sort(np.random.randint(0,len(range(n_rois[0])),num_rois)),
        2: np.sort(np.random.randint(0,len(range(n_rois[1])),num_rois)),
    }
else:
    if roi_idx is not None:
        dmd_rois = {
            1: np.arange(0,n_rois[0])[roi_idx[0]],
            2: np.arange(0,n_rois[1])[roi_idx[1]]
        }
    else:
        dmd_rois = {
            1: np.arange(0,n_rois[0])[:num_rois],
            2: np.arange(0,n_rois[1])[:num_rois]
        }
fig, ax, traces = plot_selected_dff_rois(
    session_trace_h5,
    dmd_rois=dmd_rois,
    window_sec=(0, 300),      # absolute session time in seconds
    signal="dff",
    max_points=80_000,
    time_unit="s",
    spacing_multiplier=2,
    width=10,
    roi_height=0.25,
    title="Selected dendrite ROI dFF traces",
    save_path=SAVE_PATH,
    save_name="selected_dff_rois",
)

In [ ]:
windows = {
    "long": (30, 60),
    "medium": (45, 55),
    "short": (50,53),
}

for label, window_sec in windows.items():
    fig, ax, _ = plot_selected_dff_rois(
        session_trace_h5,
        dmd_rois=dmd_rois,
        window_sec=window_sec,
        signal="dff",
        max_points=80_000,
        time_unit="s" if (window_sec[1] - window_sec[0]) <= 180 else "min",
        spacing_multiplier=2,
        width=3.5,
        roi_height=0.35,
        title=None,
        save_path=SAVE_PATH,
        save_name=f"selected_dff_rois_{label}",
    )

### Plot selected DMD1/DMD2 ROIs with image-presentation epochs

The extraction QC stores stimulus onsets in **HARP time**, whereas the
`voltage_session_traces_*.h5` timebase may be either absolute HARP time or a
relative/nominal trace clock. The cells below use
`behavior/imaging_epochs.csv` to map each image and omission into the H5
timebase, including separate epoch-wise mapping for interrupted acquisitions.

In [ ]:
# Seven image identities plus omission.
# Colors are assigned in image_h5_name_map order from extraction QC.
im_colors = [
    "#c5cae9", "#ffcdd2", "#c8e6c9", "#ffe0b2",
    "#e1bee7", "#d7ccc8",
    "#9fd3f2",   # final image: distinct light blue
    "#d9d9d9",   # omission: light gray
]

In [ ]:
import json
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd


def _filename_stem(path_string):
    """Path.stem that also handles Windows separators on non-Windows systems."""
    return Path(str(path_string).replace("\\", "/")).stem


def _read_reference_timebase(h5_path, dmd=1):
    with h5py.File(h5_path, "r") as h5:
        key = f"DMD{int(dmd)}"
        if key not in h5:
            key = next(k for k in h5.keys() if str(k).upper().startswith("DMD"))
        return np.asarray(h5[key]["timebase_sec"][:], dtype=float), key


def _choose_event_metadata(per_dmd, preferred_dmd=1):
    preferred = f"DMD{int(preferred_dmd)}"
    candidates = [preferred] + [k for k in per_dmd if k != preferred]

    for key in candidates:
        payload = per_dmd.get(key, {})
        if payload.get("skipped", False):
            continue
        if "stimulus_onsets_used_for_extraction" in payload:
            return key, payload

    raise KeyError(
        "No non-skipped DMD contains stimulus_onsets_used_for_extraction"
    )


def _samples_per_epoch(reconstruction_metadata, epochs):
    raw = reconstruction_metadata.get("samples_per_epoch", {})
    counts = []

    for row in epochs.itertuples():
        candidates = (
            str(int(row.epoch_index)),
            int(row.epoch_index),
            f"epoch_{int(row.epoch_index)}",
        )
        value = next((raw[k] for k in candidates if k in raw), None)
        counts.append(None if value is None else int(value))

    return counts


def _make_harp_to_trace_mapper(trace_t, epochs, reconstruction_metadata):
    """
    Build an epoch-aware mapper from HARP seconds to the H5 trace timebase.

    The session-trace H5 may use:
      1) absolute HARP time, or
      2) a relative/nominal trace clock beginning near zero.

    For case (2), HARP time is mapped linearly within each imaging epoch.
    samples_per_epoch from extraction QC is used when available.
    """
    trace_t = np.asarray(trace_t, dtype=float)
    epochs = epochs.sort_values("epoch_index").reset_index(drop=True)

    harp_start = float(epochs["start_time"].iloc[0])
    harp_end = float(epochs["end_time"].iloc[-1])
    session_duration = harp_end - harp_start
    tolerance = max(0.25, 0.001 * session_duration)

    absolute_harp = (
        abs(trace_t[0] - harp_start) <= tolerance
        and abs(trace_t[-1] - harp_end) <= tolerance
    )

    if absolute_harp:
        epoch_map = epochs.assign(
            trace_start=epochs["start_time"].astype(float),
            trace_end=epochs["end_time"].astype(float),
        )
        mode = "absolute_harp"
    else:
        counts = _samples_per_epoch(reconstruction_metadata, epochs)
        counts_valid = (
            all(c is not None and c > 0 for c in counts)
            and abs(sum(counts) - len(trace_t)) <= max(2, len(counts))
        )

        trace_bounds = []
        if counts_valid:
            i0 = 0
            for count in counts:
                i1 = min(i0 + count, len(trace_t))
                if i1 <= i0:
                    raise ValueError("Invalid samples_per_epoch in extraction QC")
                trace_bounds.append((float(trace_t[i0]), float(trace_t[i1 - 1])))
                i0 = i1
        else:
            if len(epochs) > 1:
                warnings.warn(
                    "samples_per_epoch could not be reconciled with the H5 "
                    "timebase; allocating trace time across epochs in proportion "
                    "to HARP epoch duration."
                )
            durations = epochs["duration_s"].to_numpy(float)
            edges = np.r_[
                trace_t[0],
                trace_t[0]
                + np.cumsum(durations / durations.sum())
                * (trace_t[-1] - trace_t[0]),
            ]
            trace_bounds = list(zip(edges[:-1], edges[1:]))

        epoch_map = epochs.copy()
        epoch_map["trace_start"] = [b[0] for b in trace_bounds]
        epoch_map["trace_end"] = [b[1] for b in trace_bounds]
        mode = "epoch_scaled_relative"

    def mapper(harp_times):
        harp_times = np.asarray(harp_times, dtype=float)
        mapped = np.full(harp_times.shape, np.nan, dtype=float)

        for row in epoch_map.itertuples():
            mask = (
                (harp_times >= float(row.start_time))
                & (harp_times <= float(row.end_time))
            )
            if not np.any(mask):
                continue

            harp_fraction = (
                (harp_times[mask] - float(row.start_time))
                / (float(row.end_time) - float(row.start_time))
            )
            mapped[mask] = (
                float(row.trace_start)
                + harp_fraction * (float(row.trace_end) - float(row.trace_start))
            )

        return mapped

    return mapper, mode, epoch_map


def load_image_presentation_spans(
    asset,
    session_trace_h5,
    *,
    im_colors,
    image_duration_sec=0.25,
    omission_duration_sec=None,
    preferred_dmd=1,
    qc_filename="voltage_extraction_qc_dff_robust_f0_trial.json",
):
    """
    Load image and omission onsets from voltage extraction QC and align them
    to the session-trace H5 using behavior/imaging_epochs.csv.

    Returns
    -------
    spans : pandas.DataFrame
        One row per image presentation/omission, with HARP and H5 trace times.
    legend_colors : dict
        Image label -> color, in extraction order.
    alignment : dict
        Alignment mode and epoch mapping used for QC.
    """
    if omission_duration_sec is None:
        omission_duration_sec = image_duration_sec

    qc_json = Path(asset.qc_dir) / "voltage" / qc_filename
    epochs_csv = Path(asset.qc_dir) / "behavior" / "imaging_epochs.csv"

    if not qc_json.exists():
        raise FileNotFoundError(qc_json)
    if not epochs_csv.exists():
        raise FileNotFoundError(epochs_csv)

    with open(qc_json, "r", encoding="utf-8") as f:
        extraction_qc = json.load(f)

    epochs = pd.read_csv(epochs_csv)
    required = {"epoch_index", "start_time", "end_time", "duration_s"}
    missing = required.difference(epochs.columns)
    if missing:
        raise ValueError(f"imaging_epochs.csv is missing columns: {sorted(missing)}")

    dmd_key, dmd_qc = _choose_event_metadata(
        extraction_qc["per_dmd"],
        preferred_dmd=preferred_dmd,
    )
    onsets = dmd_qc["stimulus_onsets_used_for_extraction"]
    reconstruction_metadata = dmd_qc.get("reconstruction_metadata", {})

    trace_t, trace_dmd_key = _read_reference_timebase(
        session_trace_h5,
        dmd=preferred_dmd,
    )
    mapper, alignment_mode, epoch_map = _make_harp_to_trace_mapper(
        trace_t,
        epochs,
        reconstruction_metadata,
    )

    image_map = dmd_qc.get("image_h5_name_map", {})
    if image_map:
        image_paths = [image_map[k] for k in sorted(image_map)]
    else:
        image_paths = list(onsets["image_identity"])

    n_images = len(image_paths)
    if len(im_colors) < n_images + 1:
        raise ValueError(
            f"Need at least {n_images + 1} colors "
            f"({n_images} images plus omission); received {len(im_colors)}"
        )

    image_colors = dict(zip(image_paths, im_colors[:n_images]))
    omission_color = im_colors[n_images]
    rows = []

    for image_path in image_paths:
        for onset in onsets["image_identity"].get(image_path, []):
            rows.append(
                {
                    "event_type": "image",
                    "image_path": image_path,
                    "image_name": _filename_stem(image_path),
                    "harp_start_sec": float(onset),
                    "harp_end_sec": float(onset) + float(image_duration_sec),
                    "color": image_colors[image_path],
                }
            )

    for onset in onsets.get("omission", []):
        rows.append(
            {
                "event_type": "omission",
                "image_path": None,
                "image_name": "omission",
                "harp_start_sec": float(onset),
                "harp_end_sec": float(onset) + float(omission_duration_sec),
                "color": omission_color,
            }
        )

    spans = pd.DataFrame(rows).sort_values("harp_start_sec").reset_index(drop=True)
    spans["trace_start_sec"] = mapper(spans["harp_start_sec"].to_numpy())
    spans["trace_end_sec"] = mapper(spans["harp_end_sec"].to_numpy())

    n_before = len(spans)
    spans = spans.dropna(subset=["trace_start_sec", "trace_end_sec"]).reset_index(drop=True)
    if len(spans) < n_before:
        warnings.warn(
            f"Dropped {n_before - len(spans)} events outside the detected imaging epochs."
        )

    legend_colors = {
        _filename_stem(path): color for path, color in image_colors.items()
    }
    legend_colors["omission"] = omission_color

    alignment = {
        "event_metadata_dmd": dmd_key,
        "trace_timebase_dmd": trace_dmd_key,
        "mode": alignment_mode,
        "epoch_map": epoch_map,
        "trace_time_range_sec": (float(trace_t[0]), float(trace_t[-1])),
    }

    return spans, legend_colors, alignment

In [ ]:
# --------------------------- USER CONTROLS ---------------------------

# Any number of ROIs can be selected independently from the two DMDs.
DMD_ROIS = {
    1: [0, 1, 2, 3],
    2: [0, 1, 2, 3],
}

# Window is expressed in the H5 trace timebase.
# The plotted x-axis is relative, so this produces a 0–15 s display.
WINDOW_START_SEC = 30.0
WINDOW_DURATION_SEC = 15.0
WINDOW_SEC = (
    WINDOW_START_SEC,
    WINDOW_START_SEC + WINDOW_DURATION_SEC,
)

IMAGE_DURATION_SEC = 0.25
OMISSION_DURATION_SEC = 0.25

SHOW_IMAGE_LEGEND = True
SAVE_FIGURE = False

session_trace_h5 = (
    Path(SESSION_TRACE_H5)
    if SESSION_TRACE_H5 is not None
    else (
        Path(asset.derived_dir)
        / "voltage"
        / f"voltage_session_traces_{TRACE_VARIANT}.h5"
    )
)

image_spans, image_legend, image_alignment = load_image_presentation_spans(
    asset,
    session_trace_h5,
    im_colors=im_colors,
    image_duration_sec=IMAGE_DURATION_SEC,
    omission_duration_sec=OMISSION_DURATION_SEC,
    preferred_dmd=next(iter(DMD_ROIS)),
    qc_filename=f"voltage_extraction_qc_{TRACE_VARIANT}.json",
)

# Restrict to events that overlap the requested display window.
window_spans = image_spans.loc[
    (image_spans["trace_end_sec"] >= WINDOW_SEC[0])
    & (image_spans["trace_start_sec"] <= WINDOW_SEC[1])
].copy()

print("Trace H5:", session_trace_h5)
print("Alignment mode:", image_alignment["mode"])
print(
    f"Plotting {len(window_spans)} image/omission spans "
    f"within {WINDOW_SEC[0]:.2f}–{WINDOW_SEC[1]:.2f} s"
)
display(image_alignment["epoch_map"])

In [ ]:
save_name = (
    f"{asset.session_id}_selected_rois_images_"
    f"{WINDOW_SEC[0]:.1f}-{WINDOW_SEC[1]:.1f}s"
)

fig, ax, plotted_traces = plot_selected_dff_rois(
    session_trace_h5,
    dmd_rois=DMD_ROIS,
    window_sec=WINDOW_SEC,
    signal=SIGNAL,
    event_spans=window_spans,
    image_legend=image_legend if SHOW_IMAGE_LEGEND else None,
    event_alpha=0.30,
    trace_colors={1: "#e99c81", 2: "#7bbcd5"},
    max_points=80_000,
    time_unit="s",
    center="median",
    spacing_multiplier=1.35,
    dmd_gap=0.9,
    lw=0.60,
    width=10,
    roi_height=0.34,
    title=None,
    show_scale_bar=True,
    save_path=SAVE_PATH if SAVE_FIGURE else None,
    save_name=save_name if SAVE_FIGURE else None,
)